# Cross-Staff Calibration Campaign 001

**Notebook:** 04 Empirical Calibration  
**Survey:** SUR-CAL-2026-001  
**Purpose:** Estimate the effective width of each fiducial and determine whether a one-parameter geometric calibration improves agreement with the observations.  
**Author:** Dennis Hazelett

## 1. Scientific Question

The ideal geometry notebook treated each fiducial's nominal width as exact. This notebook asks whether the observer–instrument system behaves as though each fiducial has a slightly different **effective width**.

The model remains physically motivated. It does not fit an arbitrary curve.

## 2. Calibration Model

For target angular width $\theta$ and effective fiducial width $F_{\mathrm{eff}}$, the model predicts

$$L = \frac{F_{\mathrm{eff}}}{2\tan(\theta/2)},$$

where $L$ is the observed staff reading.

A separate value of $F_{\mathrm{eff}}$ is estimated for each nominal fiducial.

## 3. Import Packages

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit

plt.rcParams['figure.figsize'] = (8, 5)

## 4. Load the Canonical Survey

In [ ]:
# Find the repository root by walking upward until data/ is found.
repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / 'data').exists():
    repo_root = repo_root.parent

survey_dir = repo_root / 'data' / 'examples' / 'SUR-CAL-2026-001'

with open(survey_dir / 'survey.json', encoding='utf-8') as f:
    survey = json.load(f)

observations = pd.read_csv(survey_dir / 'observations.csv')
notes = pd.read_csv(survey_dir / 'notes.csv')

print(f"Loaded {len(observations)} observations from {survey['survey_id']}")

## 5. Prepare Analysis Columns

In [ ]:
cal = observations.copy()

aliases = {
    'calibration.target_distance': 'target_distance',
    'calibration.target_width': 'target_width',
    'calibration.target_id': 'target_id',
}

for source, alias in aliases.items():
    if source in cal.columns:
        cal[alias] = cal[source]

required = ['observation_id', 'fiducial_id', 'staff_reading', 'target_distance', 'target_width']
missing = [column for column in required if column not in cal.columns]
if missing:
    raise KeyError(f'Missing required analysis columns: {missing}')

for column in ['fiducial_id', 'staff_reading', 'target_distance', 'target_width']:
    cal[column] = pd.to_numeric(cal[column], errors='raise')

cal['target_angle_rad'] = 2 * np.arctan(
    cal['target_width'] / (2 * cal['target_distance'])
)

cal.head()

## 6. Define the Physical Model

The independent variable is target angle. The only fitted parameter is effective fiducial width.

In [ ]:
def predicted_staff_reading(theta_rad, effective_fiducial_width):
    return effective_fiducial_width / (2 * np.tan(theta_rad / 2))

## 7. Fit Each Fiducial Separately

In [ ]:
fit_rows = []

for fiducial, subset in cal.groupby('fiducial_id'):
    theta = subset['target_angle_rad'].to_numpy()
    observed = subset['staff_reading'].to_numpy()

    popt, pcov = curve_fit(
        predicted_staff_reading,
        theta,
        observed,
        p0=[fiducial],
        maxfev=10000,
    )

    effective_width = popt[0]
    standard_error = np.sqrt(np.diag(pcov))[0] if pcov.size else np.nan

    predicted = predicted_staff_reading(theta, effective_width)
    residuals = observed - predicted

    fit_rows.append({
        'nominal_fiducial_width': fiducial,
        'effective_fiducial_width': effective_width,
        'standard_error': standard_error,
        'observation_count': len(subset),
        'mean_residual': residuals.mean(),
        'residual_sd': residuals.std(ddof=1) if len(residuals) > 1 else np.nan,
        'rmse': np.sqrt(np.mean(residuals ** 2)),
    })

fit_summary = pd.DataFrame(fit_rows).sort_values('nominal_fiducial_width')
fit_summary

Interpret these values cautiously. A fitted effective width may reflect not only physical dimensions, but also sighting, focus, alignment, and observer judgment.

## 8. Compare Nominal and Effective Widths

In [ ]:
fit_summary['difference'] = (
    fit_summary['effective_fiducial_width']
    - fit_summary['nominal_fiducial_width']
)

fit_summary['percent_difference'] = (
    100 * fit_summary['difference']
    / fit_summary['nominal_fiducial_width']
)

fit_summary

In [ ]:
plt.scatter(
    fit_summary['nominal_fiducial_width'],
    fit_summary['effective_fiducial_width'],
)

limits = [
    min(
        fit_summary['nominal_fiducial_width'].min(),
        fit_summary['effective_fiducial_width'].min(),
    ),
    max(
        fit_summary['nominal_fiducial_width'].max(),
        fit_summary['effective_fiducial_width'].max(),
    ),
]
plt.plot(limits, limits, linestyle='--')
plt.xlabel('Nominal Fiducial Width')
plt.ylabel('Estimated Effective Fiducial Width')
plt.title('Nominal vs Effective Fiducial Width')
plt.tight_layout()
plt.show()

## 9. Add Calibrated Predictions and Residuals

In [ ]:
width_lookup = fit_summary.set_index(
    'nominal_fiducial_width'
)['effective_fiducial_width']

cal['effective_fiducial_width'] = cal['fiducial_id'].map(width_lookup)
cal['calibrated_staff_reading'] = predicted_staff_reading(
    cal['target_angle_rad'],
    cal['effective_fiducial_width'],
)
cal['calibrated_residual'] = (
    cal['staff_reading'] - cal['calibrated_staff_reading']
)

cal[
    [
        'observation_id',
        'fiducial_id',
        'staff_reading',
        'calibrated_staff_reading',
        'calibrated_residual',
    ]
].head()

## 10. Observed vs Calibrated Prediction

In [ ]:
plt.scatter(cal['calibrated_staff_reading'], cal['staff_reading'])

limits = [
    min(cal['calibrated_staff_reading'].min(), cal['staff_reading'].min()),
    max(cal['calibrated_staff_reading'].max(), cal['staff_reading'].max()),
]
plt.plot(limits, limits, linestyle='--')
plt.xlabel('Calibrated Predicted Staff Reading')
plt.ylabel('Observed Staff Reading')
plt.title('Observed vs Calibrated Prediction')
plt.tight_layout()
plt.show()

## 11. Residuals Across the Operating Range

In [ ]:
for fiducial, subset in cal.groupby('fiducial_id'):
    plt.scatter(
        subset['calibrated_staff_reading'],
        subset['calibrated_residual'],
        label=fiducial,
    )

plt.axhline(0, linestyle='--')
plt.xlabel('Calibrated Predicted Staff Reading')
plt.ylabel('Residual')
plt.title('Calibrated Residuals Across the Instrument Range')
plt.legend(title='Nominal Fiducial')
plt.tight_layout()
plt.show()

## 12. Compare Ideal and Calibrated Models

In [ ]:
cal['ideal_staff_reading'] = predicted_staff_reading(
    cal['target_angle_rad'],
    cal['fiducial_id'],
)
cal['ideal_residual'] = cal['staff_reading'] - cal['ideal_staff_reading']

model_comparison = pd.DataFrame({
    'model': ['ideal geometry', 'effective-width calibration'],
    'mean_residual': [
        cal['ideal_residual'].mean(),
        cal['calibrated_residual'].mean(),
    ],
    'residual_sd': [
        cal['ideal_residual'].std(ddof=1),
        cal['calibrated_residual'].std(ddof=1),
    ],
    'rmse': [
        np.sqrt(np.mean(cal['ideal_residual'] ** 2)),
        np.sqrt(np.mean(cal['calibrated_residual'] ** 2)),
    ],
})

model_comparison

The calibrated model uses more fitted parameters than the ideal model, so a lower in-sample RMSE is expected. This comparison is descriptive, not a formal test of generalization.

## 13. Inspect the Largest Remaining Residuals

In [ ]:
largest_remaining = (
    cal.assign(absolute_residual=cal['calibrated_residual'].abs())
    .sort_values('absolute_residual', ascending=False)
)

largest_remaining[
    [
        'observation_id',
        'fiducial_id',
        'target_width',
        'target_distance',
        'staff_reading',
        'calibrated_staff_reading',
        'calibrated_residual',
        'notes_id',
    ]
].head(10)

## 14. Save Derived Outputs

In [ ]:
output_dir = repo_root / 'analysis' / 'tables' / 'cross-staff-calibration-001'
output_dir.mkdir(parents=True, exist_ok=True)

fit_summary.to_csv(
    output_dir / 'effective-fiducial-widths.csv',
    index=False,
)
model_comparison.to_csv(
    output_dir / 'geometry-model-comparison.csv',
    index=False,
)
cal[
    [
        'observation_id',
        'fiducial_id',
        'target_width',
        'target_distance',
        'staff_reading',
        'ideal_staff_reading',
        'ideal_residual',
        'effective_fiducial_width',
        'calibrated_staff_reading',
        'calibrated_residual',
        'notes_id',
    ]
].to_csv(
    output_dir / 'empirical-calibration-observations.csv',
    index=False,
)

print(f'Saved calibration outputs to: {output_dir}')

## 15. Interpretation Notes

The empirical calibration results are consistent with the conclusions of the previous notebook. The 0.25-inch and 1-inch fiducials lie very close to the ideal geometric prediction, suggesting that the nominal fiducial widths provide an excellent description of the instrument over much of its operating range.

The remaining fiducials are represented by relatively few observations. In particular, the 4-inch fiducial was observed only once and therefore provides insufficient evidence to determine whether its larger residual reflects a systematic effect or ordinary measurement variability. Additional observations should be collected before drawing conclusions about these larger fiducials.

Estimating an effective fiducial width produces a modest reduction in the overall residual spread. The improvement is measurable but not dramatic, indicating that the ideal geometric model already explains much of the observed behavior.

Inspection of the largest residuals reveals that four of the five largest deviations are associated with measurements of the 0.5-inch calibration target. This is consistent with observations made during data collection that this target approached the practical resolution limit of the instrument. 

No obvious residual structure is apparent across the operating range following empirical calibration. Within the limits of the present dataset, the instrument appears to behave consistently across the calibration configurations that could be measured reliably.

Overall, the agreement between the simple geometric model and the observations is better than anticipated. Although the instrument is not highly precise, it appears to be both fundamentally accurate and internally consistent across a broad range of target widths. Improved precision is likely achievable through repeated observations and observer experience rather than major changes to the underlying geometry.

During data collection, operating the cross-staff proved more challenging than initially expected. Maintaining stable alignment while simultaneously observing the calibration target and the fiducial is difficult because they lie at substantially different focal distances. Subjectively, the instrument becomes considerably easier to use when the crosspiece is positioned more than approximately 15 inches from the observer's eye. This perceived ergonomic limitation is not immediately apparent in the residual plots but may warrant targeted investigation in future calibration campaigns.

## 16. Limitations

This is an in-sample calibration using a small dataset. The same observations are used to estimate and evaluate each effective width. Future campaigns should provide additional repeated measurements and, ideally, an independent validation set.

## 17. Next Step

If systematic residual structure remains after effective-width calibration, the next notebook can model residual dependence on fiducial, target geometry, or operating range. If little structure remains, the physical model may already be adequate for the present instrument.